In [28]:
import os
import re
import json
import pandas as pd
import ollama

base_path = r"C:\Users\simon\OneDrive\Desktop\negotiation_arena"

model_name = "llama3"
n_runs = 3
max_turns = 6

In [29]:
scenarios_file = os.path.join(
    base_path,
    "data",
    "scenarios",
    "job_negotiation_scenarios.json"
)

conditions_file = os.path.join(
    base_path,
    "data",
    "scenarios",
    "experimental_conditions.json"
)

with open(scenarios_file, "r", encoding="utf-8") as f:
    scenarios = json.load(f)

with open(conditions_file, "r", encoding="utf-8") as f:
    conditions = json.load(f)

print(len(scenarios), "scenarios loaded")
print(len(conditions), "conditions loaded")

3 scenarios loaded
3 conditions loaded


In [30]:
def build_candidate_prompt(style):
    return f"""
You are the Candidate in a job contract negotiation.

Private constraints:
- Target salary: 90,000 USD
- Minimum acceptable salary: 85,000 USD
- Preferred working hours: 8 hours
- Maximum acceptable working hours: 9 hours

Negotiation style: {style}

Rules:
- Never accept salary below 85,000 USD.
- Never accept working hours above 9.
- If the employer offers salary >= 85,000 and working hours <= 9, explicitly accept.
- If you accept, write DECISION: accept.
- If the offer is not acceptable, write DECISION: continue.
- If no agreement seems possible, write DECISION: quit.
- Reply concisely.

Reply ONLY in this format:

MESSAGE: your short negotiation message
SALARY_OFFER: number or NONE
HOURS_OFFER: number or NONE
DECISION: continue / accept / quit
"""

In [31]:
def build_employer_prompt(style):
    return f"""
You are the Employer in a job contract negotiation.

Private constraints:
- Preferred salary offer: 75,000 USD
- Maximum salary offer: 87,000 USD
- Preferred working hours: 10 hours
- Minimum acceptable working hours: 9 hours

Negotiation style: {style}

Rules:
- Never offer more than 87,000 USD.
- Never accept working hours below 9.
- If the candidate proposes salary <= 87,000 and working hours >= 9, explicitly accept.
- If you accept, write DECISION: accept.
- If the proposal is not acceptable, write DECISION: continue.
- If no agreement seems possible, write DECISION: quit.
- Reply concisely.

Reply ONLY in this format:

MESSAGE: your short negotiation message
SALARY_OFFER: number or NONE
HOURS_OFFER: number or NONE
DECISION: continue / accept / quit
"""

In [32]:
def parse_structured_response(text):
    result = {
        "message": None,
        "salary_offer": None,
        "hours_offer": None,
        "decision": None
    }

    for line in text.splitlines():
        line = line.strip()

        if line.startswith("MESSAGE:"):
            result["message"] = line.replace("MESSAGE:", "").strip()

        elif line.startswith("SALARY_OFFER:"):
            value = line.replace("SALARY_OFFER:", "").strip()

            if value.upper() == "NONE":
                result["salary_offer"] = None
            else:
                number = re.search(r"\d[\d,]*", value)
                if number:
                    result["salary_offer"] = int(number.group().replace(",", ""))

        elif line.startswith("HOURS_OFFER:"):
            value = line.replace("HOURS_OFFER:", "").strip()

            if value.upper() == "NONE":
                result["hours_offer"] = None
            else:
                number = re.search(r"\d+(\.\d+)?", value)
                if number:
                    result["hours_offer"] = float(number.group())

        elif line.startswith("DECISION:"):
            result["decision"] = line.replace("DECISION:", "").strip().lower()

    return result

In [33]:
def is_valid_agreement_for_candidate(salary, hours):
    return (
        salary is not None
        and hours is not None
        and salary >= 85000
        and hours <= 9
    )


def is_valid_agreement_for_employer(salary, hours):
    return (
        salary is not None
        and hours is not None
        and salary <= 87000
        and hours >= 9
    )


def check_acceptance(speaker, parsed):
    salary = parsed["salary_offer"]
    hours = parsed["hours_offer"]
    decision = parsed["decision"]

    if decision != "accept":
        return False

    if speaker == "Candidate":
        return is_valid_agreement_for_candidate(salary, hours)

    if speaker == "Employer":
        return is_valid_agreement_for_employer(salary, hours)

    return False

In [34]:
def run_negotiation_simulation(
    scenario,
    condition,
    run_id,
    model_name="llama3",
    max_turns=6
):
    conversation_log = []

    candidate_prompt = build_candidate_prompt(
        condition["candidate_style"]
    )

    employer_prompt = build_employer_prompt(
        condition["employer_style"]
    )

    current_message = """
MESSAGE: I would like a salary of 90,000 USD and an 8 hour workday.
SALARY_OFFER: 90000
HOURS_OFFER: 8
DECISION: continue
"""

    outcome = None

    for turn in range(max_turns):

        employer_response = ollama.chat(
            model=model_name,
            messages=[
                {"role": "system", "content": employer_prompt},
                {"role": "user", "content": current_message}
            ]
        )

        employer_text = employer_response["message"]["content"]
        parsed_employer = parse_structured_response(employer_text)

        conversation_log.append({
            "scenario_id": scenario["scenario_id"],
            "condition": condition["condition_name"],
            "run_id": run_id,
            "turn": turn,
            "speaker": "Employer",
            "text": employer_text,
            "salary_offer": parsed_employer["salary_offer"],
            "hours_offer": parsed_employer["hours_offer"],
            "decision": parsed_employer["decision"]
        })

        if check_acceptance("Employer", parsed_employer):
            outcome = "Agreement"
            break

        if parsed_employer["decision"] == "quit":
            outcome = "Failure"
            break

        current_message = employer_text

        candidate_response = ollama.chat(
            model=model_name,
            messages=[
                {"role": "system", "content": candidate_prompt},
                {"role": "user", "content": current_message}
            ]
        )

        candidate_text = candidate_response["message"]["content"]
        parsed_candidate = parse_structured_response(candidate_text)

        conversation_log.append({
            "scenario_id": scenario["scenario_id"],
            "condition": condition["condition_name"],
            "run_id": run_id,
            "turn": turn,
            "speaker": "Candidate",
            "text": candidate_text,
            "salary_offer": parsed_candidate["salary_offer"],
            "hours_offer": parsed_candidate["hours_offer"],
            "decision": parsed_candidate["decision"]
        })

        if check_acceptance("Candidate", parsed_candidate):
            outcome = "Agreement"
            break

        if parsed_candidate["decision"] == "quit":
            outcome = "Failure"
            break

        current_message = candidate_text

    if outcome is None:
        outcome = "Timeout"

    return conversation_log, outcome

In [ ]:
all_turns = []
all_outcomes = []

for scenario in scenarios:
    for condition in conditions:
        for run_id in range(n_runs):

            print(
                f"Running scenario {scenario['scenario_id']} | "
                f"{condition['condition_name']} | run {run_id}"
            )

            log, outcome = run_negotiation_simulation(
                scenario=scenario,
                condition=condition,
                run_id=run_id,
                model_name=model_name,
                max_turns=max_turns
            )

            all_turns.extend(log)

            all_outcomes.append({
                "scenario_id": scenario["scenario_id"],
                "condition": condition["condition_name"],
                "run_id": run_id,
                "outcome": outcome,
                "n_turns": len(log)
            })

llm_turns_df = pd.DataFrame(all_turns)
llm_outcomes_df = pd.DataFrame(all_outcomes)

llm_outcomes_df

Running scenario 1 | cooperative | run 0
Running scenario 1 | cooperative | run 1
Running scenario 1 | cooperative | run 2
Running scenario 1 | competitive | run 0
Running scenario 1 | competitive | run 1
Running scenario 1 | competitive | run 2
Running scenario 1 | mixed | run 0
Running scenario 1 | mixed | run 1
Running scenario 1 | mixed | run 2
Running scenario 2 | cooperative | run 0
Running scenario 2 | cooperative | run 1
Running scenario 2 | cooperative | run 2
Running scenario 2 | competitive | run 0
Running scenario 2 | competitive | run 1
Running scenario 2 | competitive | run 2
Running scenario 2 | mixed | run 0
Running scenario 2 | mixed | run 1
Running scenario 2 | mixed | run 2
Running scenario 3 | cooperative | run 0
Running scenario 3 | cooperative | run 1


In [ ]:
llm_outcomes_df["outcome"].value_counts()

In [ ]:
llm_outcomes_df.groupby("condition")["outcome"].value_counts()

In [ ]:
llm_outcomes_df.groupby("condition")["n_turns"].mean()